# 单文件 LFP 流程

本 Notebook 可在真实 FIF 到位时运行真实文件；若路径不存在，则自动切换到明确标记的合成信号，仅用于算法验证。它不补造动物身份、给药信息或 AIMs。

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from lfp_analysis.pipeline import run_single_file
from lfp_analysis.synthetic import validate_synthetic

sample_path = Path(r'C:\Users\PC\Desktop\94\LID-94\T80\LID-T80_all_channels-epo.fif')
real_output = PROJECT_ROOT / 'results' / 'notebook_LID-T80'
if sample_path.exists():
    manifest = run_single_file(sample_path, PROJECT_ROOT / 'configs' / 'default.yaml', real_output, PROJECT_ROOT / 'metadata')
    run_mode = 'REAL FIF (local read-only input)'
else:
    manifest = validate_synthetic(PROJECT_ROOT / 'results' / 'notebook_synthetic_validation')
    run_mode = 'SYNTHETIC VALIDATION ONLY'
print(run_mode)
manifest

REAL FIF (local read-only input)


{'analysis_version': '0.1.0',
 'input_path': 'C:\\Users\\PC\\Desktop\\94\\LID-94\\T80\\LID-T80_all_channels-epo.fif',
 'input_sha256': '2751bce6cdad2a17d7ba72978b0b59221f8265dd27d9c77a8e77600ed75a0582',
 'input_size_bytes': 6722922,
 'file_id': 'LID-T80_all_channels-epo',
 'identity_status': 'file_only_identity_unresolved',
 'registry_row': {'file_id': 'LID-T80_all_channels-epo',
  'session_id': nan,
  'animal_id': nan,
  'file_path': 'C:\\Users\\PC\\Desktop\\94\\LID-94\\T80\\LID-T80_all_channels-epo.fif',
  'dose_state': 'post_dose',
  'drug': 'L-DOPA',
  'nominal_dose_time_min': '80',
  'actual_record_start': nan,
  'actual_record_end': nan,
  'preprocessing_notes': '预处理历史待核实；T80为名义给药后时间；动物身份和session未提供',
  'include_file_level': 'true',
  'is_example': 'known_sample_unresolved_identity'},
 'n_epochs': 21,
 'n_channels': 16,
 'n_times': 5000,
 'sampling_rate_hz': 1000.0,
 'epoch_tmin_s': 0.0,
 'epoch_tmax_s': 4.999,
 'effective_valid_duration_s': 105.0,
 'events_are_retained_raw_value

In [2]:
if 'REAL' in run_mode:
    quality_file = pd.read_csv(real_output / 'quality_file.csv')
    quality_channel = pd.read_csv(real_output / 'quality_channel.csv')
    epochs_trace = pd.read_csv(real_output / 'epochs_trace.csv')
    display(quality_file)
    display(quality_channel.loc[quality_channel['n_warn_epochs'] > 0])
    display(epochs_trace.loc[epochs_trace['drop_reason'].notna() & epochs_trace['drop_reason'].ne('')])
else:
    display(manifest)

,n_epochs,n_channels,n_times,sampling_rate_hz,nominal_duration_s,effective_valid_duration_s,n_fail_epoch_channel_rows,n_warn_epoch_channel_rows,n_duplicate_epoch_rows
0,21,16,5000,1000.0,105.0,105.0,0,1,0


,channel_array_index,channel_name,n_epochs,n_fail_epochs,n_warn_epochs,mean_std,max_abs,max_issue_score
12,12,TETFP21,21,0,1,0.000112,0.001986,1


,file_id,saved_index,original_candidate_index,events_raw_value,selection_value,confirmed_time_start_s,confirmed_time_end_s,quality_status,quality_flags,drop_reason,time_coordinate_note
4,LID-T80_all_channels-epo,NaN,4,NaN,NaN,NaN,NaN,dropped_upstream,NaN,USER,events raw value retained; not interpreted as ...
6,LID-T80_all_channels-epo,NaN,6,NaN,NaN,NaN,NaN,dropped_upstream,NaN,USER,events raw value retained; not interpreted as ...
8,LID-T80_all_channels-epo,NaN,8,NaN,NaN,NaN,NaN,dropped_upstream,NaN,USER,events raw value retained; not interpreted as ...
10,LID-T80_all_channels-epo,NaN,10,NaN,NaN,NaN,NaN,dropped_upstream,NaN,USER,events raw value retained; not interpreted as ...


## 解释边界

T80 是名义给药后时点；本流程不把 epoch 的 events 值解释成原始记录起止时间。没有登记动物身份、脑区映射和行为同步时，Notebook 只展示文件级结果。